In [ ]:
import pandas as pd
import pickle
import arviz as az
from plotly import graph_objects as go
from datetime import datetime
import pycountry

from emu_renewal.inputs import ANALYSIS_TYPES, get_latest_analyses
from emu_renewal.inputs import DATA_PATH
from emu_renewal.outputs import get_output_dir, load_targets
from emu_renewal.plotting import plot_spaghetti_calib_comparison, plot_post_prior_comparison, plot_imm_props

In [ ]:
country = "Spain"
analysis = "no_mob"

In [ ]:
time = get_latest_analyses(country, ANALYSIS_TYPES)[analysis]
outdir = get_output_dir(country, analysis, time)

In [ ]:
spaghetti = pd.read_hdf(outdir / "spaghetti.h5", key="spaghetti")

In [ ]:
spaghetti = pd.read_hdf(outdir / "spaghetti.h5")
targets = load_targets(country, analysis, time)
idata = az.from_netcdf(outdir / "idata.nc")

In [ ]:
mob = pd.read_csv(DATA_PATH / f"mobility/{pycountry.countries.get(name=country).alpha_3}_gmob_data.csv", index_col=0)
mob.index = pd.to_datetime(mob.index)
non_resi_mob = mob.loc[:, mob.columns!="residential"]
model_mob = non_resi_mob.mean(axis=1).rolling(7).mean().dropna()
analysis_start = datetime(2020, 9, 1)
analysis_end = datetime(2021, 6, 15)
filtered_mob = model_mob.loc[(analysis_start < model_mob.index) & (model_mob.index < analysis_end)]

In [ ]:
outputs = list(targets.keys()) + ["process", "seropos"]
spagh_plot = plot_spaghetti_calib_comparison(spaghetti, targets, outputs)
spagh_plot.add_traces(go.Scatter(x=filtered_mob.index, y=filtered_mob, line={"color": "blue"}), rows=outputs.index("process") + 1, cols=1)

In [ ]:
plot_imm_props(spaghetti, 2)

In [ ]:
priors = pickle.load(open(outdir / "priors.pkl", "rb"))
epi_params = [p for p in priors if p != "shared_dispersion"]
plot_post_prior_comparison(idata, epi_params, priors, req_grid=[8, 2], req_size=[10, 40]);